In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import time
from multiprocessing import Pool
from tqdm.auto import tqdm
import re
from copy import deepcopy

import numpy as np
from scipy import integrate
from matplotlib import pyplot as plt

import noctiluca as nl
import bayesmsd

/home/sgh/gitlibs/chromatin_dynamics/.venv_SD_py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
filename = '/data/sgh/science/2024_minflux/20260106_chromatin_dynamics_all_data.h5'
data       = nl.io.load.hdf5(filename)['data']

In [3]:
n_subsample = 4 # cut off the "kink" at the beginning of MINFLUX data
def subsample(traj):
    out = nl.Trajectory(traj[::n_subsample])
    out.meta['Δt'] = n_subsample*traj.meta['Δt']
    return out

In [4]:
data.makeSelection('minflux')
data.apply(subsample, inplace=True)

# Fits

In [5]:
def chop(traj, dt=None, L=200, Fmin=2):
    if dt is None:
        dt = traj.meta['Δt']
    
    def chop_traj(traj, dt=dt):
        if 'Δt' in traj.meta:
            dt = traj.meta['Δt']
            
        chops = []
        i0 = 0
        while i0 < len(traj):
            i1 = i0+L
            chop = traj.data[:, i0:min(i1, len(traj)), :]
            try:
                t_start = np.nonzero(~np.any(np.isnan(chop), axis=(0, 2)))[0][0]
            except IndexError: # no valid entries in this chop
                new_traj = nl.Trajectory(chop[:, [0]])
            else:
                new_traj = nl.Trajectory(chop[:, t_start:])
                
            new_traj.meta['Δt'] = dt
            chops.append(new_traj)

            i0 = i1
            
        return chops
    
    chops = chop_traj(traj)
    out = nl.TaggedSet(chops, hasTags=False)
    while len(chops) > 1:
        cg_traj = nl.Trajectory(np.stack([traj.data[:, 0] for traj in chops], axis=1))
        cg_traj.meta['Δt'] = L*chops[0].meta['Δt']
        chops = chop_traj(cg_traj)
        for traj in chops:
            out.add(traj)
    
    # Clean out useless trajectories
    out.makeSelection(lambda traj, _: traj.F < Fmin)
    out.deleteSelection()
    return out

In [6]:
ct = 'RPE'
bar = tqdm()

fits = {}
for treatment in ['ctrl']:

    cond = ['H2B', ct, treatment]
    fits[treatment] = {
        'single' : {},
        'joints' : {},
    }

    # Minflux
    data.makeSelection(['minflux', *cond], logic=all)
    dt = data[0].meta['Δt']

    fitdata = nl.TaggedSet()
    for traj in data:
        fitdata |= chop(traj.rescale(1e6, keepmeta=['Δt']))

    with nl.Parallelize():
        _ = nl.analysis.MSD(fitdata, chunksize=10, show_progress=True)

    fit = bayesmsd.lib.NPFit(fitdata, motion_blur_f=dt/n_subsample, parametrization='(log(αΓ), α)')
    fit.parameters['log(σ²) (dim 1)'].fix_to = 'log(σ²) (dim 0)'
    fit.likelihood_chunksize = 200

    fits[treatment]['single'][f'minflux'] = fit

    bar.update()

    # Conventional
    for dt_tag in ['100ms', '2s']:
        data.makeSelection(['SPT', dt_tag, *cond], logic=all)
        dt = data[0].meta['Δt']
        tau_e = 0.08671 # same exposure for both conditions

        fitdata = data.apply(lambda traj : traj.relative(keepmeta=['MSD', 'Δt']), inplace=False)

        fit = bayesmsd.lib.NPFit(fitdata, motion_blur_f=tau_e, parametrization='(log(αΓ), α)')
        fit.parameters['log(σ²) (dim 1)'].fix_to = 'log(σ²) (dim 0)'
        fit.likelihood_chunksize = 100

        fits[treatment]['single'][f'SPT-{dt_tag}'] = fit

        bar.update()

    # Assemble list of fit(group)s to run
    groups = {
        'minflux'       : ['minflux'],
        'SPT 100ms'     : ['SPT-100ms'],
        'SPT 2s'        : ['SPT-2s'],
        'SPT'           : ['SPT-100ms', 'SPT-2s'],
        'minflux + SPT' : ['minflux', 'SPT-100ms', 'SPT-2s'],
    }

    for groupname in groups:
        fits_dict = fits[treatment]['single']

        fit = bayesmsd.FitGroup({name : fits_dict[name] for name in groups[groupname]})
        fit.parameters['α']       = deepcopy(fits_dict['minflux'].parameters[      'α (dim 0)'])
        fit.parameters['log(αΓ)'] = deepcopy(fits_dict['minflux'].parameters['log(αΓ) (dim 0)'])

        # hacky...
        def patch_initial_params(self=fit):
            params = type(self).initial_params(self)
            a    = [val for key, val in params.items() if      'α' in key][0]
            logG = [val for key, val in params.items() if 'log(αΓ)' in key][0]
            params['α'] = a
            params['log(αΓ)'] = logG
            return params
        fit.initial_params = patch_initial_params

        for fitname in fit.fits_dict:
            fit.parameters[fitname+f' α (dim 0)'].fix_to = 'α'
            if fitname == 'minflux':
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = 'log(αΓ)'
            else: # not minflux, so correct for 2-loc
                def twoGref(params): return params['log(αΓ)']+np.log(2)
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = twoGref

        fits[treatment]['joints'][groupname] = fit

        bar.update()

bar.close()

0it [00:00, ?it/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████| 2013/2013 [00:00<00:00, 2134.47it/s]
8it [00:04,  1.87it/s]


In [7]:
fitres = {}
for treatment in ['ctrl']:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    fitres[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)

        with nl.Parallelize():
            fitres[treatment][name] = fits[treatment]['joints'][name].run(show_progress=True)

        for key in fitres[treatment][name]['params']:
            print(key, fitres[treatment][name]['params'][key])
        print()


||   RPE ctrl  ||

minflux


fit iterations: 77it [00:26,  2.95it/s]


minflux log(σ²) (dim 0) -8.471580847770893
α 0.32670378913128806
log(αΓ) -6.107139156232378
minflux α (dim 0) 0.32670378913128806
minflux log(αΓ) (dim 0) -6.107139156232378

SPT 100ms


fit iterations: 67it [00:26,  2.55it/s]


SPT-100ms log(σ²) (dim 0) -7.330500667282927
α 0.3236176018650647
log(αΓ) -6.167796156244741
SPT-100ms α (dim 0) 0.3236176018650647
SPT-100ms log(αΓ) (dim 0) -5.474648975684795

SPT 2s


...

SPT-2s log(σ²) (dim 0) -14.323971070134153
α 0.2921009448153289
log(αΓ) -6.243645794214902
SPT-2s α (dim 0) 0.2921009448153289
SPT-2s log(αΓ) (dim 0) -5.550498613654956

SPT


fit iterations: 182it [01:53,  1.60it/s]


SPT-100ms log(σ²) (dim 0) -7.384517606024676
SPT-2s log(σ²) (dim 0) -18.158456578717768
α 0.29335932852189545
log(αΓ) -6.234882374343765
SPT-100ms α (dim 0) 0.29335932852189545
SPT-2s α (dim 0) 0.29335932852189545
SPT-100ms log(αΓ) (dim 0) -5.541735193783819
SPT-2s log(αΓ) (dim 0) -5.541735193783819

minflux + SPT


fit iterations: 320it [04:36,  1.16it/s]

minflux log(σ²) (dim 0) -8.57591264941107
SPT-100ms log(σ²) (dim 0) -7.349087982057732
SPT-2s log(σ²) (dim 0) -16.582425919821027
α 0.30146848695248274
log(αΓ) -6.229420141451138
minflux α (dim 0) 0.30146848695248274
minflux log(αΓ) (dim 0) -6.229420141451138
SPT-100ms α (dim 0) 0.30146848695248274
SPT-2s α (dim 0) 0.30146848695248274
SPT-100ms log(αΓ) (dim 0) -5.536272960891193
SPT-2s log(αΓ) (dim 0) -5.536272960891193



In [8]:
nl.io.write.hdf5(fitres, f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')

## Profiler
Estimate credible intervals for point estimates from profile likelihood. __Attention: computationally expensive__

This can also move the point estimate, if we find better parameters while exploring

In [9]:
fitres = nl.io.load.hdf5(f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')
mci = {}
for treatment in ['ctrl']:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    mci[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)
        
        profiler = bayesmsd.Profiler(fits[treatment]['joints'][name], max_restarts=50)
        profiler.point_estimate = fitres[treatment][name]

        with nl.Parallelize():
            mci[treatment][name] = profiler.find_MCI(show_progress=True)

        for key in mci[treatment][name]:
            m, (cil, cih) = mci[treatment][name][key]
            print(f"{key:>25s} = {m:>6.3f} [{cil:>6.3f}, {cih:>6.3f}]")
        print()


||   RPE ctrl  ||

minflux


profiler iterations: 78it [10:30,  8.08s/it]


  minflux log(σ²) (dim 0) = -8.472 [-8.516, -8.430]
                        α =  0.327 [ 0.315,  0.339]
                  log(αΓ) = -6.107 [-6.170, -6.043]

SPT 100ms


profiler iterations: 82it [11:55,  8.73s/it]


SPT-100ms log(σ²) (dim 0) = -7.331 [-7.365, -7.297]
                        α =  0.324 [ 0.313,  0.334]
                  log(αΓ) = -6.168 [-6.186, -6.150]

SPT 2s


...

[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 446
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445


profiler iterations: 195it [1:53:18, 56.46s/it]

[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 446
[bayesmsd.Fit]  BadCovarianceError: Cholesky factorization failed, dpotrf returned 445


profiler iterations: 311it [2:54:05, 33.59s/it]

  minflux log(σ²) (dim 0) = -8.576 [-8.605, -8.550]
SPT-100ms log(σ²) (dim 0) = -7.349 [-7.365, -7.333]
   SPT-2s log(σ²) (dim 0) = -32.582 [  -inf, -10.420]
                        α =  0.301 [ 0.299,  0.304]
                  log(αΓ) = -6.229 [-6.235, -6.225]



In [10]:
nl.io.write.hdf5(mci, f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')